# Path-IO: TCGA-LUAD / TCGA-LUSC end-to-end inference

**Interactive reproducibility tutorial — Modules 1–3.** This notebook starts with a GDC download manifest, processes H&E whole-slide images, predicts eight-class habitat maps, packs slides by participant, extracts 7-channel patches, runs the four released CNN checkpoints, and applies the locked Fisher–RSF pipeline to produce continuous patient-level risk scores.

**Scope:** TCGA is not a matched immunotherapy-treated metastatic NSCLC validation cohort. These scores demonstrate locked-model inference, **not** validated treatment benefit, calibrated survival probability, or clinical utility. The optional CT + clinical composite model (Module 4) is **not runnable from H&E-only TCGA inputs**.

**Run location:** Open this notebook with your working directory set to the **PATH-IO repository root**. Each stage runs your repository scripts; no patient survival data or model retraining is required. Begin with 1–3 slides to check storage, runtime and image metadata.

## 0. Software and checkpoints

Use a Python environment with PyTorch, numpy, pandas, Pillow, OpenSlide (including its native system library), timm, scikit-learn, scikit-survival, shapely, joblib, and dependencies documented by Modules 1–3. Install the official [GDC Data Transfer Tool](https://gdc.cancer.gov/access-data/gdc-data-transfer-tool) separately. The **UNI backbone**, fitted **eight-class classifier**, **four selected CNN checkpoints**, and **locked four-CNN Fisher–RSF artifact** are required. Download released CNN weights from [PATH-IO Releases]((pretrained models available by controlled access)) and keep them in `3.Survival_prediction/weights/`. Verify model availability and licensing before use.

This notebook does not download model weights automatically or fabricate checkpoint names. Loading `.joblib`/`.pt` files should only be done from trusted sources.

In [ ]:
from pathlib import Path
import sys, shutil, subprocess, csv, re
import pandas as pd
import numpy as np

ROOT = Path.cwd().resolve()  # must be the PATH-IO repository root
PROJECTS = ["TCGA-LUAD"]  # change to ["TCGA-LUAD", "TCGA-LUSC"] to run both
WORK = ROOT / "data" / "tcga_demo"
RESULTS = ROOT / "results" / "tcga_demo"
DEVICE = "cuda:0"  # change to "cpu" if no CUDA GPU
N_SLIDES = 3  # initial demo; set None for all prepared slides
UNI_CHECKPOINT = ROOT / "models" / "UNI" / "pytorch_model.bin"
UNI_CLASSIFIER = ROOT / "models" / "UNI" / "UNI_8class_classifier.joblib"
CHECKPOINT_CSV = ROOT / "5.TCGA_inference_tutorial" / "checkpoints.csv"
FISHER_RSF = ROOT / "3.Survival_prediction" / "weights" / "OS_locked_Fisher_RSF_pipeline.joblib"

MOD1 = ROOT / "1.Patch_extraction_WSI_habitat_map_generation"
MOD2 = ROOT / "2.WSI_map_preprocessing"
MOD3 = ROOT / "3.Survival_prediction"
TUTORIAL = ROOT / "5.TCGA_inference_tutorial"
WORK.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)
print("Projects:", PROJECTS)

In [ ]:
required_scripts = [
    MOD1 / "patch_extraction" / "svs2tile.py",
    MOD1 / "inference" / "predict_tissue_classes.py",
    MOD1 / "habitat_map_generation" / "generate_habitat_maps.py",
    MOD2 / "tissue_packing.py", MOD2 / "generate_patches.py",
    MOD3 / "models.py", MOD3 / "predict_fisher_rsf.py",
    TUTORIAL / "prepare_tcga_slides.py", TUTORIAL / "infer_released_cnns.py"
]
missing = [str(x) for x in required_scripts if not x.is_file()]
if missing:
    raise FileNotFoundError("Repository files missing. Adjust ROOT or folder names:\n" + "\n".join(missing))
print("All expected scripts found.")

def run(*args):
    cmd = [str(a) for a in args]
    print("\n$", " ".join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)


## 1. Download public TCGA slide images

In the [NCI GDC Portal](https://portal.gdc.cancer.gov/), filter for **TCGA-LUAD** or **TCGA-LUSC**, *Biospecimen → Slide Image*, and **diagnostic tumor slides** where available. Review sample type and open-access status, add `.svs` slides to your cart, and download the **manifest**. Save it as `data/tcga_demo/TCGA-LUAD_manifest.txt` (or `TCGA-LUSC_manifest.txt`).

The GDC client downloads files into UUID-named directories. Whole-slide images can consume substantial storage. The next cell only runs if the manifest exists; no silent substitution of data occurs.

In [ ]:
GDC_CLIENT = shutil.which("gdc-client")
print("gdc-client:", GDC_CLIENT or "not installed / not on PATH")
for project in PROJECTS:
    manifest = WORK / f"{project}_manifest.txt"
    print(project, "manifest:", manifest, "exists:", manifest.exists())
# Uncomment after installing gdc-client and downloading your manifest(s):
# for project in PROJECTS:
#     manifest = WORK / f"{project}_manifest.txt"
#     if not manifest.is_file(): raise FileNotFoundError(manifest)
#     if not GDC_CLIENT: raise RuntimeError("Install gdc-client first")
#     run(GDC_CLIENT, "download", "-m", manifest, "-d", WORK / project / "gdc_download")

## 2. Prepare slide folders and check barcode grouping

This step collects `.svs` files from the GDC UUID subfolders, preserving slide filenames. TCGA patient grouping uses the **12-character participant barcode**, not the default split-at-first-hyphen rule (which would incorrectly group all slides as `TCGA`). Keep LUAD and LUSC separate.

In [ ]:
for project in PROJECTS:
    gdc_root = WORK / project / "gdc_download"
    if not gdc_root.is_dir():
        print(f"SKIP {project}: place downloaded GDC slides in {gdc_root}")
        continue
    run(sys.executable, TUTORIAL / "prepare_tcga_slides.py",
        "--gdc-root", gdc_root,
        "--output", WORK / project / "slides",
        "--project", project)
    slides = sorted((WORK / project / "slides").glob("*.svs"))
    print(project, "slides:", len(slides), "examples:", [s.name for s in slides[:3]])

## 3. Extract WSI image tiles (Module 1)

The supplied `svs2tile.py` has **hardcoded input/output roots**. The next cell writes a temporary copy with those two roots substituted; the repository source file remains unchanged. At Aperio 20×, it extracts 256×256 regions; at Aperio 40×, it extracts 512×512 regions and downsamples to 256×256. It **skips slides** without `aperio.AppMag` exactly `20` or `40`. This is nonoverlapping WSI tiling, not stride-128 tiling.

**Caution:** Whole-slide extraction is expensive; start with a few slides. Check the number of output patch folders after running.

In [ ]:
extractor = MOD1 / "patch_extraction" / "svs2tile.py"
source = extractor.read_text(encoding="utf-8")
old_in = 'slide_rootpath = "/path/to/wsi_directory/"'
old_out = 'out_rootpath = "/path/to/output_directory/"'
if source.count(old_in) != 1 or source.count(old_out) != 1:
    raise RuntimeError("svs2tile.py path settings differ from the supplied version; inspect before running")
for project in PROJECTS:
    slides_dir = WORK / project / "slides"
    slides = sorted(slides_dir.glob("*.svs"))
    if not slides:
        print("SKIP", project, "no slides")
        continue
    patches_dir = WORK / project / "patches"
    modified = source.replace(old_in, f"slide_rootpath = {str(slides_dir)!r}")
    modified = modified.replace(old_out, f"out_rootpath = {str(patches_dir)!r}")
    temp_script = WORK / f"svs2tile_{project}.py"
    temp_script.write_text(modified, encoding="utf-8")
    stop = len(slides) if N_SLIDES is None else min(N_SLIDES, len(slides))
    run(sys.executable, temp_script, "0", str(stop))
    print(project, "slide patch folders:", len(list(patches_dir.glob("*"))))

## 4. Classify patches into eight tissue compartments (Module 1)

Run the released UNI backbone and eight-class classifier on extracted image patches. The eight classes are Background, Bronchi, Inflammatory, Lung, Necrosis, Stroma, Tumor, Vessel. The inference workflow uses raw patches **without Macenko normalization**.

In [ ]:
for checkpoint in (UNI_CHECKPOINT, UNI_CLASSIFIER):
    if not checkpoint.is_file():
        print("MISSING model asset:", checkpoint)
# Run after placing both required model assets:
# for project in PROJECTS:
#     patches_dir = WORK / project / "patches"
#     if not any(patches_dir.iterdir()): continue
#     run(sys.executable, MOD1 / "inference" / "predict_tissue_classes.py",
#         "--patch-root", patches_dir,
#         "--output", WORK / project / "tissue_predictions",
#         "--uni-checkpoint", UNI_CHECKPOINT,
#         "--classifier", UNI_CLASSIFIER,
#         "--device", DEVICE, "--batch-size", "64", "--workers", "4")

## 5. Generate slide-level habitat maps (Module 1)

The original postprocessing can filter predicted tissue labels using matching registered GeoJSON tissue contours at a 1% overlap threshold. **No GeoJSON is supplied by default in this TCGA demonstration**, so the script exports the predicted maps without contour filtering. This is a methodological difference from contour-filtered analysis.

In [ ]:
for project in PROJECTS:
    pred_dir = WORK / project / "tissue_predictions"
    if not pred_dir.is_dir() or not list(pred_dir.rglob("*_prediction_256.csv")):
        print("SKIP", project, "no prediction CSVs")
        continue
    run(sys.executable, MOD1 / "habitat_map_generation" / "generate_habitat_maps.py",
        "--csv-root", pred_dir,
        "--output", WORK / project / "habitat_postprocessed",
        "--final-maps", WORK / project / "final_maps",
        "--workers", "4")
    print(project, "maps:", len(list((WORK / project / "final_maps").glob("*.png"))))

## 6. Pack multiple WSIs per TCGA participant (Module 2)

Compactly pack tissue maps **without resizing** or changing class colors. The explicit TCGA participant-ID regex prevents incorrect grouping. This preserves the 2-pixel internal gap, 4-pixel between-map gap, 4-pixel margin, rotation setting and target aspect ratio 1.5 from the original packing procedure.

In [ ]:
for project in PROJECTS:
    maps = WORK / project / "final_maps"
    if not maps.is_dir() or not list(maps.glob("*.png")):
        print("SKIP", project, "no habitat maps")
        continue
    run(sys.executable, MOD2 / "tissue_packing.py",
        "--input", f"{project}={maps}",
        "--output", WORK / "patient_maps",
        "--patient-id-regex", r"^(TCGA-[A-Za-z0-9]{2}-[A-Za-z0-9]{4})")
    print(project, "packed maps:", len(list((WORK / "patient_maps" / project).glob("*.png"))))

## 7. Extract seven-channel habitat patches (Module 2)

Extract 64×64 patches at stride 32, retaining patches with at least 10% tissue (computed over real pixels). If no patch passes the threshold, the original code retains the best nonempty patch. Each `.pt` file stores `patches` of shape `[N,7,64,64]` and `coords` of shape `[N,2]`. Background is represented by zeros across the seven channels.

In [ ]:
cache = WORK / "patch_cache"
for project in PROJECTS:
    packed = WORK / "patient_maps" / project
    if not packed.is_dir() or not list(packed.glob("*.png")):
        print("SKIP", project, "no packed maps")
        continue
    run(sys.executable, MOD2 / "generate_patches.py",
        "--input", f"{project}={packed}",
        "--output", cache,
        "--patch-size", "64", "--stride", "32", "--min-tissue", "0.10")
print("Total patient caches:", len(list(cache.glob("*.pt"))))

## 8. Register four released CNN checkpoints

Download the **selected** R18, R34, R50 and RX50 checkpoints from [GitHub Releases]((pretrained models available by controlled access)). Place them under `3.Survival_prediction/weights/` and create `5.TCGA_inference_tutorial/checkpoints.csv` with columns `model,checkpoint,column` in **R18, R34, R50, RX50** order.

The `column` values must exactly match the saved Fisher–RSF artifact's `columns` metadata. Example filenames and `E###` values are **placeholders**, not actual weights. The following cell checks compatibility and refuses to proceed on missing files or a mismatch.

In [ ]:
if CHECKPOINT_CSV.is_file() and FISHER_RSF.is_file():
    import joblib
    manifest = pd.read_csv(CHECKPOINT_CSV, dtype=str)
    assert manifest.model.tolist() == ["R18", "R34", "R50", "RX50"]
    artifact = joblib.load(FISHER_RSF)  # load only trusted model artifacts
    expected = list(artifact["columns"])
    assert manifest.column.tolist() == expected, (manifest.column.tolist(), expected)
    for raw in manifest.checkpoint:
        ck = Path(raw)
        if not ck.is_absolute(): ck = CHECKPOINT_CSV.parent / ck
        assert ck.is_file(), f"Missing checkpoint: {ck}"
    print("All four checkpoints match Fisher–RSF column metadata.")
else:
    print("Set up checkpoints.csv and the locked Fisher–RSF .joblib before CNN inference.")

## 9. Generate patch-level risks using the released CNNs (Module 3)

This uses the four **already-selected** checkpoints directly. It does not select epochs based on TCGA data and does not need training logs. Outputs one `.npz` per patient with `risk` `[N,4]`, `coords` and `columns`. The CNN script loads one model at a time to limit GPU usage.

In [ ]:
cnn_out = WORK / "cnn_patch_risks"
if CHECKPOINT_CSV.is_file() and FISHER_RSF.is_file() and list(cache.glob("*.pt")):
    run(sys.executable, TUTORIAL / "infer_released_cnns.py",
        "--module3", MOD3, "--cache", cache,
        "--checkpoints", CHECKPOINT_CSV,
        "--output", cnn_out, "--batch-size", "128", "--device", DEVICE)
else:
    print("SKIP: require patient patch caches, checkpoint manifest, and Fisher–RSF artifact")

## 10. Generate continuous patient-level Path-IO risk (Module 3)

Apply the **locked four-CNN GMM/Fisher/RSF** without fitting or updating any parameters on TCGA. `risk` is a continuous RSF risk score; `risk_z` uses the original training mean and standard deviation. Neither is a survival probability. Do not use a newly optimized TCGA cutoff as though it were a validated cutoff.

In [ ]:
output_csv = RESULTS / "tcga_pathio_risks.csv"
if FISHER_RSF.is_file() and cnn_out.is_dir() and list(cnn_out.glob("*.npz")):
    run(sys.executable, MOD3 / "predict_fisher_rsf.py",
        "--pipeline", FISHER_RSF,
        "--risk-dir", cnn_out,
        "--output", output_csv)
    display(pd.read_csv(output_csv).head(10))
else:
    print("SKIP: generate CNN risks and provide the locked Fisher–RSF model first")

## 11. Download TCGA-LUAD/LUSC clinical survival data (GDC)

**Source:** [NCI Genomic Data Commons (GDC)](https://portal.gdc.cancer.gov/) → Projects → TCGA-LUAD / TCGA-LUSC → Clinical. The GDC [Cases API](https://docs.gdc.cancer.gov/API/Users_Guide/Getting_Started/) provides public demographic, diagnosis, and follow-up fields. This cell retrieves records directly, so no separate clinical CSV is needed. Alternatively, download the clinical tables from the GDC Portal and adapt the subsequent merge.

For an illustrative **overall survival** endpoint, this notebook uses `days_to_death` for deceased patients and `days_to_last_follow_up` for living patients. Status is 1 for death and 0 for censored. Cases without a usable time are **not** assigned a fabricated survival duration. Follow-up fields can be incomplete or differ across GDC releases; confirm endpoint definitions and exclusions before publication.

Clinical records are **not required to calculate the pathology-only Path-IO risk score**. They are required to assess its association with observed survival. This does not reproduce the immunotherapy-treated cohort used to develop Path-IO.


In [ ]:
import requests
import json

def fetch_gdc_cases(project, page_size=500):
    endpoint = "https://api.gdc.cancer.gov/cases"
    filters = {"op": "in", "content": {"field": "project.project_id", "value": [project]}}
    fields = "submitter_id,project.project_id,demographic.vital_status,demographic.days_to_death,diagnoses.days_to_last_follow_up"
    cases = []
    offset = 0
    while True:
        params = {"filters": json.dumps(filters), "fields": fields, "format": "JSON",
                  "size": page_size, "from": offset}
        response = requests.get(endpoint, params=params, timeout=90)
        response.raise_for_status()
        payload = response.json()["data"]
        hits = payload["hits"]
        cases.extend(hits)
        offset += len(hits)
        if not hits or offset >= payload["pagination"]["total"]:
            break
    return cases

def valid_day(value):
    try:
        x = float(value)
        return x if np.isfinite(x) and x >= 0 else np.nan
    except (ValueError, TypeError):
        return np.nan

def make_survival_row(case):
    patient_id = case["submitter_id"][:12]
    demographic = case.get("demographic") or {}
    vital = str(demographic.get("vital_status") or "").strip().lower()
    death_days = valid_day(demographic.get("days_to_death"))
    followup_days = [valid_day(d.get("days_to_last_follow_up"))
                     for d in case.get("diagnoses", [])]
    followup_days = [x for x in followup_days if np.isfinite(x)]
    last_followup = max(followup_days) if followup_days else np.nan
    event = 1 if vital == "dead" else 0 if vital == "alive" else np.nan
    days = death_days if event == 1 else last_followup if event == 0 else np.nan
    return {"case_ID": patient_id, "OS_days": days, "OS_event": event,
            "vital_status": vital, "project": (case.get("project") or {}).get("project_id")}

clinical_rows = []
for project in PROJECTS:
    records = fetch_gdc_cases(project)
    clinical_rows.extend(make_survival_row(c) for c in records)
    print(project, "cases:", len(records))

clinical = pd.DataFrame(clinical_rows)
clinical.to_csv(RESULTS / "tcga_clinical_survival.csv", index=False)
print("Survival records with usable OS time and status:",
      int((clinical.OS_days.notna() & clinical.OS_event.notna()).sum()))
display(clinical.head())


## 12. Match clinical outcomes to pathology risk scores

TCGA case identifiers use the first 12 characters of the TCGA barcode (`TCGA-XX-YYYY`). The code below checks the prediction CSV columns before joining and saves a matched table. If the model emits multiple slides per patient, aggregate only according to the released pipeline's patient-level rules—**do not average risk scores arbitrarily**.


In [ ]:
if output_csv.is_file():
    risk_table = pd.read_csv(output_csv)
    print("Risk CSV columns:", list(risk_table.columns))
    id_candidates = ["case_ID", "patient_id", "patient", "MRN", "sample_id"]
    id_col = next((c for c in id_candidates if c in risk_table.columns), None)
    if id_col is None:
        raise ValueError("Set id_col to the TCGA patient identifier column in the risk CSV")
    risk_table["case_ID"] = risk_table[id_col].astype(str).str.extract(
        r"(TCGA-[A-Za-z0-9]{2}-[A-Za-z0-9]{4})", expand=False).str.upper()
    if risk_table["case_ID"].isna().any():
        raise ValueError("Some risk identifiers do not contain a valid TCGA case barcode")
    if risk_table["case_ID"].duplicated().any():
        raise ValueError("Duplicate patient risks: resolve patient-level aggregation before merging")
    matched = risk_table.merge(clinical, on="case_ID", how="left", validate="one_to_one")
    matched.to_csv(RESULTS / "tcga_pathio_risk_with_clinical.csv", index=False)
    evaluable = matched.loc[(matched.OS_days > 0) & matched.OS_event.isin([0, 1])]
    print("Matched patients:", len(matched), "Evaluable survival records:", len(evaluable))
    display(matched.head())
else:
    print("SKIP: generate pathology risk predictions before matching clinical records")


## 13. Evaluate discrimination and continuous-risk association (no Kaplan–Meier analysis)

Use observed overall-survival time and event status from GDC to calculate Harrell’s concordance index and a **univariable Cox proportional-hazards** hazard ratio (HR), 95% confidence interval, and Wald P-value **per one standard deviation increase in predicted risk**. The HR and P-value are not generated by the RSF itself. No Kaplan–Meier curves, high/low grouping, or log-rank tests are included. Results are exploratory in TCGA and do not establish immunotherapy-specific validity.

The code evaluates LUAD and LUSC separately, and optionally reports a combined descriptive estimate. It reports sample sizes and event counts and excludes missing/invalid records.


In [ ]:
# Requires: pip install lifelines scikit-survival
from lifelines import CoxPHFitter
from sksurv.metrics import concordance_index_censored

matched_file = RESULTS / "tcga_pathio_risk_with_clinical.csv"
if not matched_file.exists():
    print("SKIP: first generate and match the pathology risk and GDC clinical data.")
else:
    matched = pd.read_csv(matched_file)
    # Inspect columns before changing this value if your Module 3 output uses another name.
    risk_candidates = ["risk_score", "PathIO_risk", "pathio_risk", "predicted_risk", "risk", "RSF_risk"]
    risk_col = next((c for c in risk_candidates if c in matched.columns), None)
    if risk_col is None:
        raise ValueError(f"Set risk_col to the continuous RSF risk column; found: {list(matched.columns)}")

    rows = []
    for cohort, frame in [("TCGA-LUAD", matched[matched.project == "TCGA-LUAD"]),
                          ("TCGA-LUSC", matched[matched.project == "TCGA-LUSC"]),
                          ("Combined (descriptive)", matched)]:
        d = frame[["OS_days", "OS_event", risk_col]].copy()
        d.columns = ["time", "event", "risk"]
        d = d.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        d = d[(d.time > 0) & d.event.isin([0, 1])]
        n, events = len(d), int(d.event.sum())
        result = {"cohort": cohort, "N": n, "events": events,
                  "C_index": np.nan, "HR_per_SD": np.nan,
                  "CI95_lower": np.nan, "CI95_upper": np.nan, "Cox_Wald_p": np.nan}
        if n < 10 or events < 5 or d.risk.std(ddof=0) <= 0:
            print(f"{cohort}: insufficient evaluable observations/events or no risk variation")
            rows.append(result)
            continue
        result["C_index"] = float(concordance_index_censored(
            d.event.astype(bool).to_numpy(), d.time.to_numpy(), -d.risk.to_numpy())[0])
        d["risk_z"] = (d.risk - d.risk.mean()) / d.risk.std(ddof=0)
        cox = CoxPHFitter()
        try:
            cox.fit(d[["time", "event", "risk_z"]], duration_col="time", event_col="event")
            summary = cox.summary.loc["risk_z"]
            result.update(HR_per_SD=float(summary["exp(coef)"]),
                          CI95_lower=float(summary["exp(coef) lower 95%"]),
                          CI95_upper=float(summary["exp(coef) upper 95%"]),
                          Cox_Wald_p=float(summary["p"]))
        except Exception as exc:
            print(f"{cohort}: Cox model could not be fitted: {exc}")
        rows.append(result)
    metrics = pd.DataFrame(rows)
    metrics.to_csv(RESULTS / "tcga_pathio_survival_metrics.csv", index=False)
    display(metrics)


## 14. QC and interpretation

Check that each processed slide has patches and a habitat map, that patient barcodes were grouped correctly, and that no patient with zero tissue was assigned a fabricated score. Preserve the GDC manifest and slide inventory for provenance. TCGA slide scanning formats, tissue types, stages and treatment histories differ from the training cohorts. The predictions are **technical inference outputs**, not evidence of immunotherapy-specific prognostic or predictive performance.

For Module 4 composite inference, you would additionally need matched **CT-derived risk scores** and all required **clinical predictors** for the same patients; the H&E-only TCGA tutorial intentionally stops at the pathology-based risk score.

In [ ]:
for project in PROJECTS:
    d = WORK / project
    print(f"{project}: slides={len(list((d/'slides').glob('*.svs')))}, "
          f"WSI_patch_folders={len([p for p in (d/'patches').glob('*') if p.is_dir()])}, "
          f"habitat_maps={len(list((d/'final_maps').glob('*.png')))}, "
          f"packed_maps={len(list((WORK/'patient_maps'/project).glob('*.png')))}")
print("Patient caches:", len(list(cache.glob("*.pt"))))
print("CNN NPZ files:", len(list(cnn_out.glob("*.npz"))))
print("Final risk CSV exists:", output_csv.is_file())